In [1]:
%%writefile app.py
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
import soundfile as sf
import numpy as np
from flask import Flask, request, send_file
import tempfile

app = Flask(__name__)

# --- 1. 모델 클래스 및 전처리 함수 정의 (학습 코드와 동일) ---
class ResBlock(nn.Module):
    def __init__(self, channels):
        super(ResBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels)
        )
    def forward(self, x):
        return F.leaky_relu(x + self.conv(x), 0.2)

class UpgradedUNet(nn.Module):
    def __init__(self):
        super(UpgradedUNet, self).__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(1, 64, 3, padding=1), nn.LeakyReLU(0.2))
        self.res1 = ResBlock(64)
        self.down1 = nn.Conv2d(64, 128, 4, stride=2, padding=1)
        self.res2 = ResBlock(128)
        self.down2 = nn.Conv2d(128, 256, 4, stride=2, padding=1)
        self.res3 = ResBlock(256)
        self.up2 = nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1)
        self.dec2 = nn.Sequential(nn.Conv2d(256, 128, 3, padding=1), ResBlock(128))
        self.up1 = nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1)
        self.dec1 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), ResBlock(64))
        self.final = nn.Conv2d(64, 1, 3, padding=1)

    def forward(self, x):
        e1 = self.res1(self.enc1(x))
        e2 = self.res2(F.leaky_relu(self.down1(e1), 0.2))
        e3 = self.res3(F.leaky_relu(self.down2(e2), 0.2))
        d2 = self.up2(e3)
        if d2.size() != e2.size(): d2 = F.interpolate(d2, size=e2.shape[2:])
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        if d1.size() != e1.size(): d1 = F.interpolate(d1, size=e1.shape[2:])
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        mask = torch.sigmoid(self.final(d1))
        return x * mask, mask

def get_spec(audio):
    window = torch.hann_window(512).to(audio.device)
    spec = torch.stft(audio.squeeze(1), n_fft=512, hop_length=160, window=window, return_complex=True, center=True)
    return torch.abs(spec), torch.angle(spec)

def spec_to_wav(mag, phase):
    complex_spec = torch.polar(mag, phase)
    # STFT에서 사용한 것과 똑같은 해닝 윈도우를 생성하여 ISTFT에도 적용
    window = torch.hann_window(512).to(mag.device)
    return torch.istft(complex_spec, n_fft=512, hop_length=160, window=window, center=True)

# --- 2. 전역 모델 로드 (Singleton) ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UpgradedUNet().to(device)
# RunPod Volume Disk에 저장된 가중치 경로로 수정 필요
MODEL_PATH = '/content/UNet_GAN_Upgraded/upgraded_gan_model.pth'

if os.path.exists(MODEL_PATH):
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    model.load_state_dict(checkpoint['generator_state_dict'])
    model.eval()
    print(f"✅ 모델 로드 완료: {MODEL_PATH}")
else:
    print(f"⚠️ 경고: {MODEL_PATH} 가중치 파일이 없습니다.")


# --- 3. Overlap-Add 추론 및 후처리 함수 ---
def process_overlap_add(audio_np, sample_rate=16000):
    chunk_size = 64000  # 4초
    hop_size = 32000    # 2초 (50% 겹침)

    # 입력 오디오 길이가 4초 미만인 경우 패딩
    if len(audio_np) < chunk_size:
        pad_len = chunk_size - len(audio_np)
        audio_np = np.pad(audio_np, (0, pad_len), mode='constant')

    num_chunks = int(np.ceil((len(audio_np) - chunk_size) / hop_size)) + 1
    target_len = (num_chunks - 1) * hop_size + chunk_size

    if len(audio_np) < target_len:
        audio_np = np.pad(audio_np, (0, target_len - len(audio_np)), mode='constant')

    output_audio = np.zeros(target_len, dtype=np.float32)
    window_weight = np.zeros(target_len, dtype=np.float32)
    hanning_window = np.hanning(chunk_size)

    with torch.no_grad():
        for i in range(num_chunks):
            start = i * hop_size
            end = start + chunk_size
            chunk = audio_np[start:end]

            chunk_t = torch.FloatTensor(chunk).unsqueeze(0).unsqueeze(0).to(device)
            mag, phase = get_spec(chunk_t)
            mag = mag.unsqueeze(1)

            denoised_mag, _ = model(mag)
            denoised_wav = spec_to_wav(denoised_mag.squeeze(1), phase).cpu().numpy()[0]

            # 교차 페이드를 위한 해닝 윈도우 적용 후 덧셈
            output_audio[start:end] += denoised_wav * hanning_window
            window_weight[start:end] += hanning_window

    # 가중치 정규화
    output_audio = output_audio / np.where(window_weight > 1e-10, window_weight, 1.0)
    return output_audio

def apply_agc(audio_np, target_rms=0.05):
    current_rms = np.sqrt(np.mean(audio_np**2) + 1e-9)
    gain = target_rms / current_rms
    # 과도한 증폭 방지를 위해 클리핑
    gain = np.clip(gain, 0.1, 5.0)
    agc_audio = audio_np * gain
    # 최종 -1.0 ~ 1.0 사이 클리핑 (디스토션 방지)
    return np.clip(agc_audio, -1.0, 1.0)


# --- 4. Flask API 라우팅 ---
@app.route('/process_audio', methods=['POST'])
def process_audio_api():
    if 'file' not in request.files:
        return {"error": "파일이 제공되지 않았습니다."}, 400

    file = request.files['file']
    if file.filename == '':
        return {"error": "선택된 파일이 없습니다."}, 400

    try:
        # 임시 디렉토리 사용 (처리가 끝나면 자동 삭제되도록 유도)
        with tempfile.NamedTemporaryFile(delete=False, suffix='.wav') as temp_in, \
             tempfile.NamedTemporaryFile(delete=False, suffix='.wav') as temp_out:

            file.save(temp_in.name)

            # 1. 오디오 로드 및 16kHz 리샘플링
            y, sr = librosa.load(temp_in.name, sr=16000)

            # 2. 노이즈 제거 (Overlap-Add)
            denoised_y = process_overlap_add(y, sample_rate=16000)

            # 3. AGC 볼륨 평준화
            final_y = apply_agc(denoised_y)

            # 4. 결과물 저장
            sf.write(temp_out.name, final_y, 16000)

            # 5. 클라이언트에게 파일 전송 후 임시 파일 삭제
            response = send_file(temp_out.name, as_attachment=True, download_name="denoised_output.wav")

            # 전송 후 디스크 정리를 위해 os.remove 호출
            @response.call_on_close
            def cleanup():
                if os.path.exists(temp_in.name): os.remove(temp_in.name)
                if os.path.exists(temp_out.name): os.remove(temp_out.name)

            return response

    except Exception as e:
        return {"error": str(e)}, 500

if __name__ == '__main__':
    # 0.0.0.0으로 설정하여 외부 포트 포워딩 접근 허용
    app.run(host='0.0.0.0', port=5000)

Writing app.py


In [10]:
# 혹시 이미 켜져있는 서버가 있다면 종료
!pkill -f "python app.py"

# Flask 서버를 백그라운드에서 실행하고, 로그는 server.log 파일에 저장
!nohup python app.py > server.log 2>&1 &

# 서버가 켜질 때까지 3초 정도 대기
import time
time.sleep(3)
print("✅ 백그라운드에서 Flask 서버가 실행되었습니다.")

✅ 백그라운드에서 Flask 서버가 실행되었습니다.


In [12]:
# 5000번 포트를 사용 중인 프로세스의 PID(프로세스 ID)를 찾아 강제 종료(kill)
!fuser -k 5000/tcp

5000/tcp:             4261


In [6]:
# 서버 종료
# !pkill -f "python app.py"

In [3]:
import os
import random
import zipfile
import numpy as np
import librosa
import soundfile as sf
from google.colab import drive
import shutil

# 1. 드라이브 마운트
drive.mount('/content/drive', force_remount=True)

DATA_DIR = '/content/drive/MyDrive/UNet'
VOICE_ZIP = f'{DATA_DIR}/Korean_Voice/KsponSpeech_01.zip' # 첫 번째 음성 압축파일 사용
NOISE_ZIP = f'{DATA_DIR}/Clean_Noise_Zips/Cleaned_Noise_All.zip'
OUTPUT_DIR = '/content/custom_test_audio'
TEMP_NOISE_DIR = '/content/temp_test_noise'

# 출력 폴더 및 임시 폴더 초기화
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
if os.path.exists(TEMP_NOISE_DIR):
    shutil.rmtree(TEMP_NOISE_DIR)
os.makedirs(TEMP_NOISE_DIR)

# 2. 오디오 생성 함수 정의
def create_custom_test_samples(num_samples=5, sr=16000):
    print(f"테스트용 오디오 파일 {num_samples}개 생성을 시작합니다.\n")

    with zipfile.ZipFile(VOICE_ZIP, 'r') as vz, zipfile.ZipFile(NOISE_ZIP, 'r') as nz:
        # 10초(160,000 샘플) 이상인 pcm 파일 정보만 필터링
        # 16비트 pcm은 1샘플당 2바이트이므로 320,000바이트 이상
        min_bytes = sr * 10 * 2
        voice_infos = [info for info in vz.infolist() if info.filename.endswith('.pcm') and info.file_size >= min_bytes]

        if not voice_infos:
            print("조건에 맞는 10초 이상의 음성 파일이 없습니다.")
            return

        noise_names = [f for f in nz.namelist() if f.endswith('.wav')]

        for i in range(num_samples):
            # 1. 10초 이상 음성 데이터 무작위 로드 (길이 조절 없음)
            v_info = random.choice(voice_infos)
            with vz.open(v_info) as f:
                clean = np.frombuffer(f.read(), dtype=np.int16).astype(np.float32) / 32768.0

            # 2. 소음 데이터 무작위 로드
            n_name = random.choice(noise_names)
            nz.extract(n_name, TEMP_NOISE_DIR)
            n_path = os.path.join(TEMP_NOISE_DIR, n_name)

            try:
                noise, _ = librosa.load(n_path, sr=sr)
            except:
                noise = np.zeros(len(clean))

            # 3. 소음 길이가 음성 길이보다 짧으면 반복(Tile)
            if len(noise) < len(clean):
                reps = int(np.ceil(len(clean) / len(noise)))
                noise = np.tile(noise, reps)

            # 4. 덧붙여진 소음을 정확한 음성 길이에 맞게 자르기
            noise = noise[:len(clean)]

            # 5. 음성과 소음 혼합 (소음 크기를 0.5배로 조절하여 합성)
            noisy = clean + noise * 0.5

            # 6. 파일 저장
            out_noisy_path = os.path.join(OUTPUT_DIR, f'custom_test_noisy_{i+1}.wav')
            out_clean_path = os.path.join(OUTPUT_DIR, f'custom_test_clean_{i+1}.wav')

            sf.write(out_noisy_path, noisy, sr)
            sf.write(out_clean_path, clean, sr)

            print(f"[{i+1}/{num_samples}] 저장 완료 (길이: {len(clean)/sr:.2f}초)")

    print(f"\n모든 파일이 {OUTPUT_DIR} 폴더에 저장되었습니다.")

# 3. 실행 (생성할 파일 개수 지정)
create_custom_test_samples(num_samples=10)

Mounted at /content/drive
테스트용 오디오 파일 10개 생성을 시작합니다.

[1/10] 저장 완료 (길이: 17.36초)
[2/10] 저장 완료 (길이: 15.74초)
[3/10] 저장 완료 (길이: 12.02초)
[4/10] 저장 완료 (길이: 29.91초)
[5/10] 저장 완료 (길이: 15.93초)
[6/10] 저장 완료 (길이: 12.48초)
[7/10] 저장 완료 (길이: 18.42초)
[8/10] 저장 완료 (길이: 10.55초)
[9/10] 저장 완료 (길이: 19.53초)
[10/10] 저장 완료 (길이: 22.57초)

모든 파일이 /content/custom_test_audio 폴더에 저장되었습니다.


In [11]:
import requests
import os

file_name = 'custom_test_noisy_1.wav'
file_path = f'custom_test_audio/{file_name}' # Colab에 업로드한 테스트 파일 이름
url = 'http://127.0.0.1:5000/process_audio' # Colab 내부 통신 주소

if not os.path.exists(file_path):
    print("❌ 오류: 테스트할 오디오 파일이 없습니다. 파일을 업로드해주세요.")
else:
    print("서버로 파일을 전송합니다... (시간이 조금 걸릴 수 있습니다)")

    # POST 요청으로 파일 전송
    with open(file_path, 'rb') as f:
        files = {'file': f}
        response = requests.post(url, files=files)

    # 응답 확인 및 저장
    if response.status_code == 200:
        with open('result_denoised_colab_1.wav', 'wb') as out_f:
            out_f.write(response.content)
        print("✅ 처리 완료! 좌측 폴더에서 'result_denoised_colab_1.wav' 파일을 확인하세요.")
    else:
        print("❌ 에러 발생:", response.text)

    # 만약 에러가 났다면 서버 로그 확인
    # !cat server.log

서버로 파일을 전송합니다... (시간이 조금 걸릴 수 있습니다)
✅ 처리 완료! 좌측 폴더에서 'result_denoised_colab_1.wav' 파일을 확인하세요.
